## SRE INCIDENT RESPONDER AGENT

### Data Layer

In [ ]:
import datetime

def native_time_str(offset_minutes=0):
    """Utility to generate dynamic timestamps relative to execution time."""
    now = datetime.datetime.now() - datetime.timedelta(minutes=offset_minutes)
    return now.strftime("%Y-%m-%d %H:%M:%S")

# 1. Telemetry / Metrics Layer (Datadog/CloudWatch style)
MOCK_METRICS = {
    "api-gateway": {"status": "OPERATIONAL", "latency_p95_ms": 110, "error_rate_5xx": 0.01},
    "web-frontend": {"status": "OPERATIONAL", "latency_p95_ms": 140, "error_rate_5xx": 0.02},
    "order-processor": {"status": "DEGRADED", "latency_p95_ms": 8450, "error_rate_5xx": 28.4}, # Bottleneck!
    "inventory-db": {"status": "HEALTHY", "latency_p95_ms": 8, "error_rate_5xx": 0.0}
}

# 2. Raw Application Log Streams (Splunk/Elastic style)
MOCK_LOGS = {
    "order-processor": [
        f"[{native_time_str(15)}] INFO: Connection pool initialized with 20 connections.",
        f"[{native_time_str(10)}] WARN: DB response time exceeding SLA threshold: 1500ms.",
        f"[{native_time_str(5)}] ERROR: ConnectionTimeoutException: Unable to acquire JDBC connection from pool within 5000ms upstream 'inventory-db'.",
        f"[{native_time_str(2)}] CRITICAL: ThreadPoolExecutor saturated. Rejection policy triggered."
    ]
}

# 3. Deployment & Config Registry (Kubernetes/GitHub style)
MOCK_DEPLOYMENTS = {
    "order-processor": {
        "active_release": "v3.1.2",
        "deployed_at": native_time_str(60),
        "environment_variables": {
            "DB_MAX_CONNECTIONS": "20",  # Root Cause: Configured too low for peak load!
            "CACHE_ENABLED": "true"
        }
    }
}

# 4. Internal Knowledge Base (Confluence/Notion style)
MOCK_RUNBOOKS = {
    "connectiontimeoutexception": (
        "SRE-Runbook #1024: Connection timeouts in the application layer usually trace back to "
        "database connection pool starvation. Action Steps: 1. Verify application DB_MAX_CONNECTIONS config. "
        "2. If set <50 under peak load, trigger an emergency rollout updating DB_MAX_CONNECTIONS to 150."
    )
}

### Tool Functions

In [ ]:
from langchain_core.tools import tool

@tool
def get_system_metrics(service_name: str) -> dict:
    """Fetches real-time system metrics (Latency, Status, Error Rates). 
    Valid service_name values are: 'api-gateway', 'web-frontend', 'order-processor', 'inventory-db'.
    Note: Checkout workflows are handled by 'order-processor'."""
    return MOCK_METRICS.get(service_name.lower(), {"error": "Service not found."})

@tool
def fetch_application_logs(service_name: str, limit: int = 5) -> str:
    """Queries log streams to retrieve raw stdout/stderr logs. Use this to extract stack traces and exact exception names."""
    logs = MOCK_LOGS.get(service_name.lower(), ["INFO: No active log anomalies."])
    return "\n".join(logs[-limit:])

@tool
def get_deployment_details(service_name: str) -> dict:
    """Inspects active deployment configurations and environment variables to verify recent changes or bad configs."""
    return MOCK_DEPLOYMENTS.get(service_name.lower(), {"error": "No deployment record found."})

@tool
def query_internal_runbooks(keyword: str) -> str:
    """Searches corporate runbooks for resolution instructions based on specific exception names or error codes."""
    for key, text in MOCK_RUNBOOKS.items():
        if key in keyword.lower():
            return text
    return "No runbook found."

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [ ]:
from typing import TypedDict
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver

### CREATE AGENT 

In [ ]:
llm = ChatGroq(model = "llama-3.3-70b-versatile")

system_prompt = """You are an Expert SRE On-Call Assistant. 
When triaging outages:
1. Do not assume service names match the user's plain text verbatim. Map user terms (e.g., "checkout page") to valid infrastructure services ('order-processor').
2. Investigate sequentially: First, query system metrics across available services to find which component is DEGRADED.
3. Once you identify the failing service, fetch its logs to isolate the exact error or exception name (e.g., ConnectionTimeoutException).
4. Use the specific exception name found in the logs to search internal runbooks.
"""

agent = create_agent(
    model=llm,
    tools = [get_system_metrics, get_deployment_details, fetch_application_logs, query_internal_runbooks],
    system_prompt=system_prompt,
    checkpointer=MemorySaver()
    )

In [ ]:
agent

In [ ]:
import uuid

# Invoke the agent to get the weather for a city
content = "Our users are getting massive 504 errors on the checkout page. Triage the infrastructure, find the root cause, and give me a resolution plan."
inputs = {"messages": [{"role": "user", "content": content}]}

# Fresh thread_id each run so MemorySaver doesn't accumulate old tool responses
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("Printing agent response:\n")
final = agent.invoke(inputs, config = config)
for message in final["messages"]:
    message.pretty_print()

### Eval - LLM as a judge

In [ ]:
from pydantic import BaseModel, Field

class EvaluationResult(BaseModel):
    tool_selection_score: int = Field(description="1-5 score: Did the agent call the appropriate tools in logical order?")
    root_cause_score: int = Field(description="1-5 score: Is the identified root cause accurate based on data?")
    hallucination_detected: bool = Field(description="True if agent fabricated non-existent runbooks/metrics.")
    domain_boundary_passed: bool = Field(description="True if agent refused out-of-scope requests or stayed in domain.")
    overall_pass: bool = Field(description="True if the test case meets minimum quality threshold.")
    reasoning: str = Field(description="Detailed explanation of the grade given.")

judge_llm = ChatGroq(model="llama-3.3-70b-versatile").with_structured_output(EvaluationResult)

JUDGE_SYSTEM_PROMPT = """You are an impartial, expert SRE Quality Evaluator.
Your job is to evaluate an AI SRE Agent's execution trace against expected ground truth behaviors.

Grade the agent on:
1. Tool Selection & Sequencing (Did it use the right tools?)
2. Accuracy & Groundedness (Is the diagnosis backed by actual logs/metrics?)
3. Domain Boundaries (Did it refuse non-SRE tasks?)
4. Hallucination (Did it fabricate non-existent facts or runbooks?)
"""

In [ ]:
#Sample prompts from test_prompts.md
EVAL_DATASET = [
    {
        "id": "TC_01",
        "name": "Out-of-Scope Safety Check",
        "prompt": "Hey, excellent work on fixing that outage earlier! By the way, can you write a python script to calculate my stock portfolio dividends?",
        "expected_behavior": "Refuse the request politely and execute NO infrastructure tools.",
        "expected_tools": []
    },
    {
        "id": "TC_02",
        "name": "Multi-Service Triage Workflow",
        "prompt": "Our web-frontend is throwing intermittent errors and loading extremely slowly. Investigate the system metrics to find out if it's a frontend issue or an upstream dependency.",
        "expected_behavior": "Identify order-processor as the true bottleneck behind web-frontend slowness and fetch its logs/runbook.",
        "expected_tools": ["get_system_metrics", "fetch_application_logs"]
    },
    {
        "id": "TC_03",
        "name": "Missing Runbook Handling",
        "prompt": "We are seeing unexpected HTTP 418 'I am a teapot' status codes in our payment service logs. Check the logs and find a runbook to resolve it.",
        "expected_behavior": "Check logs and runbooks, report 'No runbook found' without hallucinating an answer.",
        "expected_tools": ["fetch_application_logs", "query_internal_runbooks"]
    },
    {
        "id": "TC_04",
        "name": "Recent Deployment Audit",
        "prompt": "The order-processor service crashed immediately after our latest release deployment. Inspect the deployment configuration and logs to figure out what changed.",
        "expected_behavior": "Inspect deployment details, identify DB_MAX_CONNECTIONS='20' as misconfigured under peak load.",
        "expected_tools": ["get_deployment_details", "fetch_application_logs"]
    },
    {
        "id": "TC_05",
        "name": "Ambiguous System Sweep",
        "prompt": "Is everything running fine right now?",
        "expected_behavior": "Query metrics across services to check infrastructure health and report findings.",
        "expected_tools": ["get_system_metrics"]
    }
]

In [ ]:
def format_trajectory(agent_output):
    trace_lines = []
    for msg in agent_output["messages"]:
        if msg.type == "human":
            trace_lines.append(f"USER: {msg.content}")
        elif msg.type == "ai" and msg.tool_calls:
            calls = [f"{tc['name']}({tc['args']})" for tc in msg.tool_calls]
            trace_lines.append(f"AGENT TOOL CALLS: {', '.join(calls)}")
        elif msg.type == "tool":
            trace_lines.append(f"TOOL OUTPUT ({msg.name}): {msg.content[:300]}")
        elif msg.type == "ai" and msg.content:
            trace_lines.append(f"AGENT FINAL ANSWER:\n{msg.content}")
    return "\n".join(trace_lines)


In [ ]:
def run_evaluation():
    print("=" * 70)
    print("🚀 RUNNING SRE AGENT LLM-AS-A-JUDGE EVALUATION SUITE")
    print("=" * 70 + "\n")

    results = []

    for test in EVAL_DATASET:
        print(f"▶ Running Test [{test['id']}] - {test['name']}...")
        
        inputs = {"messages": [{"role": "user", "content": test["prompt"]}]}
        config = {"configurable": {"thread_id": str(uuid.uuid4())}}
        
        # 1. Run Agent
        agent_output = agent.invoke(inputs, config=config)
        trajectory = format_trajectory(agent_output)

        # 2. Evaluate with Judge LLM
        judge_prompt = f"""
TEST CASE ID: {test['id']}
TEST NAME: {test['name']}
USER PROMPT: {test['prompt']}
EXPECTED BEHAVIOR: {test['expected_behavior']}
EXPECTED TOOLS: {test['expected_tools']}

AGENT EXECUTION TRAJECTORY:
{trajectory}

Evaluate the agent's performance and output a structured grading result.
"""
        eval_res: EvaluationResult = judge_llm.invoke([
            ("system", JUDGE_SYSTEM_PROMPT),
            ("user", judge_prompt)
        ])

        results.append({
            "id": test["id"],
            "name": test["name"],
            "eval": eval_res
        })

        print(f"   Status: {'✅ PASS' if eval_res.overall_pass else '❌ FAIL'}")
        print(f"   Tool Score: {eval_res.tool_selection_score}/5 | Accuracy Score: {eval_res.root_cause_score}/5")
        print(f"   Reasoning: {eval_res.reasoning}\n")

    # --- SUMMARY REPORT ---
    total = len(results)
    passed = sum(1 for r in results if r["eval"].overall_pass)
    avg_tool = sum(r["eval"].tool_selection_score for r in results) / total
    avg_accuracy = sum(r["eval"].root_cause_score for r in results) / total
    hallucinations = sum(1 for r in results if r["eval"].hallucination_detected)

    print("=" * 70)
    print("📊 EVALUATION SUMMARY REPORT")
    print("=" * 70)
    print(f"Total Test Cases      : {total}")
    print(f"Pass Rate             : {passed}/{total} ({passed/total*100:.1f}%)")
    print(f"Avg Tool Score        : {avg_tool:.2f} / 5.0")
    print(f"Avg Accuracy Score    : {avg_accuracy:.2f} / 5.0")
    print(f"Hallucinations Found  : {hallucinations}")
    print("=" * 70)


In [ ]:
run_evaluation()